In [17]:
import pandas as pd
from functions import get_user_number_from_config, data_from_data_sink

In [18]:
user = get_user_number_from_config()
print(user)

4


In [19]:
query = f"""
SELECT 
    sptfy_gnr_type_1,
    sptfy_gnr_type_2,
    sptfy_gnr_type_3,
    sptfy_lt_type_1,
    sptfy_lt_type_2,
    sptfy_lt_type_3
FROM dim_spotify
WHERE user_number = {user};
"""
df_spotify = data_from_data_sink(query)

/Users/dave/Desktop/BIPM/WHO_AM_I/sports-buddies/sports-buddies/functions.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


In [20]:
query = f"""
SELECT 
    gaming_type
FROM dim_steam
WHERE user_number = {user};
"""
df_steam = data_from_data_sink(query)

In [21]:
query = f"""
SELECT 
    fitness_type_strength,
    fitness_type_flexibility,
    fitness_type_relaation,
    fitness_type_endurence
FROM dim_strava
WHERE user_number = {user}
ORDER BY created_at DESC
LIMIT 1;
"""
df= data_from_data_sink(query)
# df["fitness_type"] = df.iloc[0].idxmax()
df_strava= df

In [22]:
query = f"""
SELECT 
    type_walker,
    type_sleeper
FROM dim_health
WHERE user_number = {user}
ORDER BY created_at DESC
LIMIT 1;
"""
df_health = data_from_data_sink(query)
print(df_health)

  type_walker     type_sleeper
0     typical  optimal sleeper


In [23]:
query = f"""
SELECT 
    working_type
FROM dim_linkedin
WHERE user_number = {user}
ORDER BY created_at DESC
LIMIT 1;
"""
df_linkedin= data_from_data_sink(query)


/Users/dave/Desktop/BIPM/WHO_AM_I/sports-buddies/sports-buddies/functions.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


In [24]:
df_user = pd.concat([df_health.reset_index(drop = True),df_linkedin.reset_index(drop = True),df_spotify.reset_index(drop = True),df_steam.reset_index(drop = True),df_strava.reset_index(drop = True)], axis=1)
df_user

,type_walker,type_sleeper,working_type,sptfy_gnr_type_1,sptfy_gnr_type_2,sptfy_gnr_type_3,sptfy_lt_type_1,sptfy_lt_type_2,sptfy_lt_type_3,gaming_type,fitness_type_strength,fitness_type_flexibility,fitness_type_relaation,fitness_type_endurence
0,typical,optimal sleeper,Midnight,rock_head (43.0%),electronic_addict (19.0%),mix_consumer (17.0%),weekend_prime_time_listener (68.0%),weekend_breakfast_listener (32.0%),nan (nan%),None,0.00702,0.0,0.59974,0.393241


In [25]:
def score_user_vibes_weighted_v3(user):
    def float_cast(val):
        try:
            return float(val)
        except (TypeError, ValueError):
            return 0.0
    
    def parse_genres_with_weights(raw_genres):
        cleaned = []
        weights = []
        for g in raw_genres:
            match = re.match(r"([a-zA-Z_ ]+)\s*\(([\d.]+)%\)", str(g))
            if match:
                name = match.group(1).strip()
                pct = float(match.group(2))
            else:
                name = str(g).strip()
                pct = 0.0
            cleaned.append(name)
            weights.append(pct)
        total = sum(weights)
        normalized_weights = [w / total if total > 0 else 0 for w in weights]
        return list(zip(cleaned, normalized_weights))

    raw_genres = [user.get('sptfy_gnr_type_1', [''])[0],
                  user.get('sptfy_gnr_type_2', [''])[0],
                  user.get('sptfy_gnr_type_3', [''])[0]]
    genres_with_weights = parse_genres_with_weights(raw_genres)

    lt = [user.get('sptfy_lt_type_1', [''])[0],
          user.get('sptfy_lt_type_2', [''])[0],
          user.get('sptfy_lt_type_3', [''])[0]]
    gamer = user.get('gaming_type', [None])[0]
    sleeper = user.get('type_sleeper', [None])[0]
    walker = user.get('type_walker', [None])[0]
    working = user.get('working_type', [None])[0]
    age = user.get('age', [None])[0]

    fitness_weights = {
    "strength": float_cast(user.get("fitness_type_strength")),
    "flexibility": float_cast(user.get("fitness_type_flexibility")),
    "relaxation": float_cast(user.get("fitness_type_relaation")),
    "endurance": float_cast(user.get("fitness_type_endurence")),
}

    is_very_young = age is not None and age < 22
    is_young = 22 <= age < 30
    is_mid = 30 <= age < 45
    boomer = 45 <= age <= 65
    is_senior = age is not None and age > 65

    scores = {
        "Party Explorer": 0,
        "Creative Urban Nomad": 0,
        "Efficiency Minimalist": 0,
        "Slow Living Spirit": 0,
        "Digital Couch Potato": 0,
        "Balanced Berliner": 0,
        "Mindful Creative": 0,
        "Streetwise Hustler": 0
    }

    # 🎵 Genres
    for genre, weight in genres_with_weights:
        if genre in ["electronic_addict", "hiphop_head"]:
            scores["Party Explorer"] += 2 * weight
            scores["Streetwise Hustler"] += 1 * weight
        if genre in ["indie_explorer", "jazz_purist"]:
            scores["Creative Urban Nomad"] += 2 * weight
            scores["Mindful Creative"] += 1.5 * weight
        if genre in ["classical_connoisseur", "jazz_purist"]:
            scores["Slow Living Spirit"] += 2 * weight
            scores["Mindful Creative"] += 2 * weight
        if genre in ["mix_consumer"]:
            scores["Mindful Creative"] += 1.0 * weight

    # 🎧 Listening Types
    if "weekend_party_listener" in lt:
        scores["Party Explorer"] += 2
        scores["Streetwise Hustler"] += 1
    if "late_night_listener" in lt:
        scores["Creative Urban Nomad"] += 1
    if "working_focus_listener" in lt:
        scores["Efficiency Minimalist"] += 2
        scores["Mindful Creative"] += 1
    if "weekend_breakfast_listener" in lt:
        scores["Mindful Creative"] += 1

    # ⏰ Working
    if working in ["midnight", "weekend"]:
        scores["Creative Urban Nomad"] += 1
        scores["Streetwise Hustler"] += 1
        scores["Efficiency Minimalist"] += 0.5
    if working == "early_bird":
        scores["Efficiency Minimalist"] += 2

    # 💤 Sleep
    if sleeper == "short":
        scores["Efficiency Minimalist"] += 1
    if sleeper == "optimal sleeper":
        scores["Balanced Berliner"] += 0.5
        scores["Mindful Creative"] += 1

    # 🏃 Activity
    if walker == "active":
        scores["Efficiency Minimalist"] += 1.5
        scores["Streetwise Hustler"] += 1
    if walker == "sedentary":
        scores["Digital Couch Potato"] += 2
    if walker == "typical":
        scores["Mindful Creative"] += 0.25

    # 💪 Fitness
    scores["Party Explorer"] += fitness_weights["strength"] * 1.5 + fitness_weights["endurance"] * 0.5
    scores["Creative Urban Nomad"] += fitness_weights["flexibility"] * 1.5
    scores["Slow Living Spirit"] += fitness_weights["relaxation"] * 1.0
    scores["Efficiency Minimalist"] += fitness_weights["endurance"] * 1.5 + fitness_weights["strength"] * 1.0
    scores["Digital Couch Potato"] += fitness_weights["strength"] * -0.5 + fitness_weights["endurance"] * -0.5
    scores["Balanced Berliner"] += sum(fitness_weights.values()) * 0.25
    scores["Mindful Creative"] += fitness_weights["relaxation"] * 2 + fitness_weights["flexibility"] * 1
    scores["Streetwise Hustler"] += fitness_weights["strength"] * 1.2 + fitness_weights["endurance"] * 0.5

    # 🎮 Gaming
    if gamer in ["frequent", "medium"]:
        scores["Digital Couch Potato"] += 2
        scores["Streetwise Hustler"] += 1

    # 🧓 Alter
    if is_very_young:
        scores["Party Explorer"] += 2
        scores["Streetwise Hustler"] += 1
    if is_young:
        scores["Party Explorer"] += 1
        scores["Digital Couch Potato"] += 0.5
        scores["Efficiency Minimalist"] += 0.75
    if is_mid:
        scores["Creative Urban Nomad"] += 1
        scores["Efficiency Minimalist"] += 1
    if boomer:
        scores["Creative Urban Nomad"] += 0.5
        scores["Efficiency Minimalist"] += 0.5
    if is_senior:
        scores["Slow Living Spirit"] += 1.5
        scores["Mindful Creative"] += 1.5

    # 🧘 Balanced fallback
    max_score = max([v for k, v in scores.items() if k != "Balanced Berliner"])
    avg_score = sum(scores.values()) / len(scores)
    scores["Balanced Berliner"] += max(0, (avg_score - max_score) * 1.5)

    # 🔄 Normalisierung
    total_score = sum(scores.values())
    scores = {k: round(v / total_score, 4) if total_score > 0 else 0 for k, v in scores.items()}

    # 🗺 Mapping Stadtteile
    vibe_to_hoods = {
        "Party Explorer": ["Friedrichshain", "Neukölln", "Mitte"],
        "Creative Urban Nomad": ["Kreuzberg", "Neukölln", "Moabit"],
        "Efficiency Minimalist": ["Charlottenburg", "Prenzlauer Berg", "Steglitz"],
        "Slow Living Spirit": ["Zehlendorf", "Köpenick", "Wilmersdorf"],
        "Digital Couch Potato": ["Lichtenberg", "Marzahn", "Spandau"],
        "Balanced Berliner": ["Schöneberg", "Tempelhof", "Mitte"],
        "Mindful Creative": ["Wedding", "Weißensee", "Pankow"],
        "Streetwise Hustler": ["Neukölln", "Wedding", "Kreuzberg"]
    }

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_vibe = sorted_scores[0][0]
    top_hoods = vibe_to_hoods[top_vibe]

    return {
        "scores": scores,
        "top_vibe": top_vibe,
        "recommended_hoods": top_hoods
    }


In [26]:
print(df_user)

  type_walker     type_sleeper working_type   sptfy_gnr_type_1  \
0     typical  optimal sleeper     Midnight  rock_head (43.0%)   

            sptfy_gnr_type_2      sptfy_gnr_type_3  \
0  electronic_addict (19.0%)  mix_consumer (17.0%)   

                       sptfy_lt_type_1                     sptfy_lt_type_2  \
0  weekend_prime_time_listener (68.0%)  weekend_breakfast_listener (32.0%)   

  sptfy_lt_type_3 gaming_type  fitness_type_strength  \
0      nan (nan%)        None                0.00702   

   fitness_type_flexibility  fitness_type_relaation  fitness_type_endurence  
0                       0.0                 0.59974                0.393241  


In [27]:
query = f"""
SELECT user_id FROM dim_user WHERE user_number = {user}
"""
user_id = data_from_data_sink(query).iloc[0, 0]


user_data = pd.read_csv("data/secret_info.csv")


user_data.loc[user_data["user_id"] == user_id,"date_of_birth"].iloc[0]
from datetime import datetime

# 1. String in Datum umwandeln
dob_str = "23.04.2000"
dob = datetime.strptime(dob_str, "%d.%m.%Y")

# 2. Alter berechnen
today = datetime.today()
age = (today - dob).days // 365


df_user["age"] = age

/Users/dave/Desktop/BIPM/WHO_AM_I/sports-buddies/sports-buddies/functions.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


In [28]:
import re
res = score_user_vibes_weighted_v3(df_user)


/var/folders/n9/9rvxcg4d1d15_3wh3wq9f0l40000gn/T/ipykernel_53439/2998587148.py:4: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(val)


In [29]:
vibe_to_hoods_explained = {
    "Party Explorer": {
        "hoods": ["Friedrichshain", "Neukölln", "Mitte"],
        "explanation": [
            "Friedrichshain is the natural habitat for the Party Explorer. With its pulsing nightlife, iconic clubs like Berghain, and raw street energy, it's the epicenter for dance, late nights, and youth culture.",
            "Neukölln offers a grittier, underground party vibe — from smoky bars to rooftops and hidden raves, it's where creative chaos meets beats.",
            "Mitte adds a more polished scene with trendsetting bars, pop-ups, and a constant rotation of events, perfect for social explorers and night owls."
        ]
    },
    "Creative Urban Nomad": {
        "hoods": ["Kreuzberg", "Neukölln", "Moabit"],
        "explanation": [
            "Kreuzberg is Berlin's bohemian heart. Murals, indie venues, and community hubs make it ideal for artistic nomads seeking inspiration and diversity.",
            "Neukölln fits for its multiculturalism and creative grit — think pop-up galleries and late-night jam sessions in smoky cafés.",
            "Moabit is the underdog artist district, still affordable and authentic, with hidden ateliers and a rising scene of makers, thinkers, and DIY dreamers."
        ]
    },
    "Efficiency Minimalist": {
        "hoods": ["Charlottenburg", "Prenzlauer Berg", "Steglitz"],
        "explanation": [
            "Charlottenburg blends order, culture, and stability — perfect for early risers and productivity fans. Its clean streets, wide sidewalks, and calm tempo match the minimalist vibe.",
            "Prenzlauer Berg has a health-conscious, organized feel — yoga at 6, green juice at 8, and off to coworking. A haven for structured creatives and routine lovers.",
            "Steglitz is residential, tidy, and well-connected. Ideal for those who prefer balance, clear routines, and no unnecessary distractions."
        ]
    },
    "Slow Living Spirit": {
        "hoods": ["Zehlendorf", "Köpenick", "Wilmersdorf"],
        "explanation": [
            "Zehlendorf offers leafy lanes, lakeside walks, and serenity. A place to breathe, meditate, and enjoy a slower pace of life.",
            "Köpenick, with its rivers, woods, and Altstadt, is a natural retreat from city pressure — ideal for calm, nature-connected souls.",
            "Wilmersdorf is graceful and residential. Quiet cafés, classic architecture, and relaxed tempo make it perfect for slow thinkers and readers."
        ]
    },
    "Digital Couch Potato": {
        "hoods": ["Lichtenberg", "Marzahn", "Spandau"],
        "explanation": [
            "Lichtenberg is practical and quiet — a no-frills zone where rent is low and the internet is fast. Ideal for gamers and streamers.",
            "Marzahn offers space, stability, and isolation — high-rise blocks perfect for introverts and long gaming sessions.",
            "Spandau is suburban but not sleepy — a laid-back zone for those who love peace, delivery food, and no FOMO."
        ]
    },
    "Balanced Berliner": {
        "hoods": ["Schöneberg", "Tempelhof", "Mitte"],
        "explanation": [
            "Schöneberg blends charm, diversity, and balance — lively but never overwhelming. Queer-friendly, leafy, and cultured.",
            "Tempelhof offers calm and connection — it’s centered, spacious, and socially mixed. Perfect for those who like harmony over hype.",
            "Mitte, again, appears here for its centrality — giving a little of everything without too much of anything."
        ]
    },
    "Mindful Creative": {
        "hoods": ["Wedding", "Weißensee", "Pankow"],
        "explanation": [
            "Wedding is raw, spiritual, and poetic. Old-school Berlin with yoga lofts and activist cafés hidden behind brutalist facades.",
            "Weißensee is peaceful and lakeside — ideal for reflection, journaling, and conscious living.",
            "Pankow blends calm residential energy with intellectual curiosity — think plant-based bistros and piano lessons."
        ]
    },
    "Streetwise Hustler": {
        "hoods": ["Neukölln", "Wedding", "Kreuzberg"],
        "explanation": [
            "Neukölln offers grit, hustle, and survival instinct. Ideal for side-hustlers, underground creatives, and self-made personalities.",
            "Wedding is where the street is the teacher — raw, real, and full of energy. Perfect for doers and grinders.",
            "Kreuzberg ties hustle with vision. Activism meets street smarts in a district that never sleeps and never settles."
        ]
    }
}


In [30]:
df_user["vibe"]= res['top_vibe']
print(df_user)

  type_walker     type_sleeper working_type   sptfy_gnr_type_1  \
0     typical  optimal sleeper     Midnight  rock_head (43.0%)   

            sptfy_gnr_type_2      sptfy_gnr_type_3  \
0  electronic_addict (19.0%)  mix_consumer (17.0%)   

                       sptfy_lt_type_1                     sptfy_lt_type_2  \
0  weekend_prime_time_listener (68.0%)  weekend_breakfast_listener (32.0%)   

  sptfy_lt_type_3 gaming_type  fitness_type_strength  \
0      nan (nan%)        None                0.00702   

   fitness_type_flexibility  fitness_type_relaation  fitness_type_endurence  \
0                       0.0                 0.59974                0.393241   

   age              vibe  
0   25  Mindful Creative  


In [32]:
data_from_data_sink(f"SELECT * FROM fact_raw_data WHERE user_number = 10;")

,raw_data_id,user_number,data_source,raw_json,extraction_time
0,10-1,10,spotify,{'extracted_at': '2025-06-18T19:14:44.262830Z'...,2025-07-05 21:12:45.451593+00:00
1,10-2,10,github,"{'username': 'GokanGorer', 'name': None, 'bio'...",2025-07-05 21:12:45.451593+00:00
2,10-3,10,linkedin,"{'name': 'Kontakt', 'title': 'Gökan Görer', 'l...",2025-07-05 21:12:45.451593+00:00
3,10-4,10,steam,"{'response': {'game_count': 10, 'games': [{'na...",2025-07-05 21:12:45.451593+00:00
4,10-5,10,health,"[{'type': 'HKQuantityTypeIdentifierStepCount',...",2025-07-05 21:12:45.451593+00:00


In [ ]:
data_from_data_sink(f"SELECT * FROM dim_user WHERE user_number = 6;")

/Users/dave/Desktop/BIPM/WHO_AM_I/sports-buddies/sports-buddies/functions.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,user_number,user_id,creation_time
